In [18]:
import os
import re
import zipfile
import urllib.request
from collections import Counter
from spellchecker import SpellChecker
import re

import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Dense, GRU, Dropout, Bidirectional, SpatialDropout1D
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.preprocessing import label_binarize

In [2]:
# Configure GPU - use only GPU 0
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Optionally limit GPU memory growth to avoid OOM errors
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f'Using GPU: {gpus[0].name}')
    except RuntimeError as e:
        print(e)
else:
    print('No GPU available, using CPU')

# Download NLTK resources
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer

# Set up stop words and tokenizer
stop_words = set(stopwords.words('english'))
tweet_tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)

No GPU available, using CPU


In [ ]:
# checkimg data if exist
os.path.exists('../data/GrammarandProductReviews.csv') 

True

In [5]:
df = pd.read_csv('../data/GrammarandProductReviews.csv')
df.head()

,id,brand,categories,dateAdded,dateUpdated,ean,keys,manufacturer,manufacturerNumber,name,...,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,upc
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",2017-07-25T00:52:42Z,2018-02-05T08:36:58Z,6.02537E+11,"602537205981,universalmusic/14331328,universal...",Universal Music Group / Cash Money,14331328,Pink Friday: Roman Reloaded Re-Up (w/dvd),...,NaN,0.0,5,https://redsky.target.com/groot-domain-api/v1/...,i love this album. it's very good. more to the...,Just Awesome,Los Angeles,NaN,Joshua,6.02537E+11
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",2017-07-25T05:16:03Z,2018-02-05T11:27:45Z,73416000391,lundbergorganiccinnamontoastricecakes/b000fvzw...,Lundberg,574764,Lundberg Organic Cinnamon Toast Rice Cakes,...,100209113.0,NaN,5,https://www.walmart.com/reviews/product/29775278,Good flavor. This review was collected as part...,Good,NaN,NaN,Dorothy W,73416000391
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",2017-07-25T05:16:03Z,2018-02-05T11:27:45Z,73416000391,lundbergorganiccinnamontoastricecakes/b000fvzw...,Lundberg,574764,Lundberg Organic Cinnamon Toast Rice Cakes,...,100209113.0,NaN,5,https://www.walmart.com/reviews/product/29775278,Good flavor.,Good,NaN,NaN,Dorothy W,73416000391
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",2017-07-25T16:26:19Z,2018-02-05T11:25:51Z,67981934427,"kylovesensualitypleasuregel/b00u2whx8s,0679819...",K-Y,67981934427,K-Y Love Sensuality Pleasure Gel,...,113026909.0,NaN,1,https://www.walmart.com/reviews/product/43383370,I read through the reviews on here before look...,Disappointed,NaN,NaN,Rebecca,67981934427
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",2017-07-25T16:26:19Z,2018-02-05T11:25:51Z,67981934427,"kylovesensualitypleasuregel/b00u2whx8s,0679819...",K-Y,67981934427,K-Y Love Sensuality Pleasure Gel,...,171267657.0,NaN,1,https://www.walmart.com/reviews/product/43383370,My husband bought this gel for us. The gel cau...,Irritation,NaN,NaN,Walker557,67981934427


In [17]:
# Drop rows without ratings first, "subset = filter" 
df_clean = df.dropna(subset=['reviews.rating']).copy()
df_clean['rating'] = df_clean['reviews.rating'].astype(int)

# Shift scores
score_min = df_clean['rating'].min()
# print(df_clean['rating'])
# print(score_min)
df_clean['score_shifted'] = df_clean['rating'] - score_min
# print(df_clean['score_shifted'])
df_clean.head()

,id,brand,categories,dateAdded,dateUpdated,ean,keys,manufacturer,manufacturerNumber,name,...,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,upc,last_char,ends_correctly,rating,score_shifted
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",2017-07-25T00:52:42Z,2018-02-05T08:36:58Z,6.02537E+11,"602537205981,universalmusic/14331328,universal...",Universal Music Group / Cash Money,14331328,Pink Friday: Roman Reloaded Re-Up (w/dvd),...,i love this album. it's very good. more to the...,Just Awesome,Los Angeles,NaN,Joshua,6.02537E+11,.,True,5,4
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",2017-07-25T05:16:03Z,2018-02-05T11:27:45Z,73416000391,lundbergorganiccinnamontoastricecakes/b000fvzw...,Lundberg,574764,Lundberg Organic Cinnamon Toast Rice Cakes,...,Good flavor. This review was collected as part...,Good,NaN,NaN,Dorothy W,73416000391,.,True,5,4
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",2017-07-25T05:16:03Z,2018-02-05T11:27:45Z,73416000391,lundbergorganiccinnamontoastricecakes/b000fvzw...,Lundberg,574764,Lundberg Organic Cinnamon Toast Rice Cakes,...,Good flavor.,Good,NaN,NaN,Dorothy W,73416000391,.,True,5,4
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",2017-07-25T16:26:19Z,2018-02-05T11:25:51Z,67981934427,"kylovesensualitypleasuregel/b00u2whx8s,0679819...",K-Y,67981934427,K-Y Love Sensuality Pleasure Gel,...,I read through the reviews on here before look...,Disappointed,NaN,NaN,Rebecca,67981934427,.,True,1,0
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",2017-07-25T16:26:19Z,2018-02-05T11:25:51Z,67981934427,"kylovesensualitypleasuregel/b00u2whx8s,0679819...",K-Y,67981934427,K-Y Love Sensuality Pleasure Gel,...,My husband bought this gel for us. The gel cau...,Irritation,NaN,NaN,Walker557,67981934427,.,True,1,0


### 1.EDA

#### 1.1.Do reviewers use punctuation correctly?

In [ ]:
#  Clean trailing spaces and grab the last character
# We use .astype(str) to ensure no errors with empty reviews
# To use specialized string functions (like slicing or stripping),
#  you must "unlock" the string toolkit by adding .str.
# .astype(str): Makes the data a string.
# .str: Tells Pandas: "I’m about to do a string operation on every row in this column

# .str.strip()
# This does not get rid of the last letter; it removes "whitespace" (invisible spaces, tabs, or newlines)
# from the very beginning and very end of the text.
df_clean['last_char'] = df_clean['reviews.text'].astype(str).str.strip().str[-1]
# print(df['last_char'])

# Code Piece                 Action,                   "Result on ""Great!  """
# .astype(str),        Changes type to string,          """Great!  """
# .str.strip(),        Removes end spaces,               """Great!"""
# .str[-1],            Grabs the last character,          """!"""

# Define valid punctuation
valid_punc = ['.', '!', '?']

# Check if the last character is valid
df_clean['ends_correctly'] = df['last_char'].isin(valid_punc)
# print(df['ends_correctly'])
# Get the count of those who DON'T use it correctly
incorrect_count = len(df) - df['ends_correctly'].sum()
proportion = (incorrect_count / len(df)) * 100

print(f"Number of reviews missing end punctuation: {incorrect_count}")
print(f"Percentage of incorrect punctuation: {proportion:.2f}%")

0         True
1         True
2         True
3         True
4         True
         ...  
71039     True
71040     True
71041    False
71042     True
71043     True
Name: ends_correctly, Length: 71044, dtype: bool
Number of reviews missing end punctuation: 12692
Percentage of incorrect punctuation: 17.86%


#### 1.2.Does the number of spelling errors differ by rating?

In [ ]:
spell = SpellChecker()
#re.findall(r'\b\w+\b', ...) ==> Think of this as a magnet that only picks up letters and numbers, leaving everything else (punctuation) behind.
# Input: "The food was grate!!"   ==> Output: ['the', 'food', 'was', 'grate']

def count_spelling_errors(text):
    # Removing punctuation and numbers, keeping only words so , i can check only words
    words = re.findall(r'\b\w+\b', str(text).lower())
    
    # finding which words are not in spell list which mean all the dataset
    misspelled = spell.unknown(words)
    
    return len(misspelled)



df_sample = df_clean.head(500).copy()
df_sample['spelling_errors'] = df_sample['reviews.text'].apply(count_spelling_errors)

# Group by rating and find the average
spelling_trend = df_sample.groupby('rating')['spelling_errors'].mean()

print("Average Spelling Errors by Rating:")
print(spelling_trend)

Average Spelling Errors by Rating:
rating
1    0.714286
2    0.750000
3    0.666667
4    0.517857
5    0.653495
Name: spelling_errors, dtype: float64
